# Reconcile canonical numbers (seed-42) — CPU only
Regenerates Member-3 canonical results from the **saved logits** in `humaid-nb1-outputs/logits/transformer/`. No training, no GPU. This cell block also replaces nb3's seen column (nb3 used seed-123 `A4_s123`; canonical is seed-42 `A4`).

In [ ]:
import numpy as np, json
from scipy.optimize import minimize_scalar
from sklearn.metrics import f1_score, precision_recall_fscore_support
LOGITS='humaid-nb1-outputs/logits/transformer'  # adjust if needed
LABELS=['caution_and_advice','displaced_people_and_evacuations','infrastructure_and_utility_damage',
'injured_or_dead_people','missing_or_found_people','not_humanitarian','other_relevant_information',
'requests_or_urgent_needs','rescue_volunteering_or_donation_effort','sympathy_and_support']

## Helpers

In [ ]:
def softmax(z,T=1.0):
    z=z/T; z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def ece(conf,correct,nb=15):
    b=np.linspace(0,1,nb+1); e=0.0; n=len(conf)
    for i in range(nb):
        m=(conf>b[i])&(conf<=b[i+1])
        if m.sum(): e+=m.sum()/n*abs(correct[m].mean()-conf[m].mean())
    return e
def nll(T,l,y): p=softmax(l,T); return -np.log(p[np.arange(len(y)),y]+1e-12).mean()
def load(tag,split): return (np.load(f'{LOGITS}/{tag}_{split}_logits.npy'), np.load(f'{LOGITS}/{tag}_{split}_labels.npy'))

## RoBERTa seed-42 (A4) — analysis model, SEEN

In [ ]:
tl,ty=load('A4','test'); pred=tl.argmax(1)
r_macro=f1_score(ty,pred,average='macro'); r_f1=precision_recall_fscore_support(ty,pred,labels=range(10),zero_division=0)[2]
T_rob=1.3509; tau_rob=0.8945; c=softmax(tl,T_rob).max(1); corr=(pred==ty).astype(float)
print('SEEN macro-F1',round(r_macro,4)); print('seen ECE %.4f -> %.4f'%(ece(softmax(tl,1).max(1),corr),ece(c,corr)))
acc=c>=tau_rob; print('seen cov/risk @tau','%.1f%%'%(acc.mean()*100),round((1-corr[acc]).mean(),4))

## DistilBERT seed-42 (A1) — deployment model: fit its OWN T and tau

In [ ]:
vl,vy=load('A1','val'); dl,dy=load('A1','test')
T_dis=minimize_scalar(lambda T:nll(T,vl,vy),bounds=(0.5,5),method='bounded').x
d_macro=f1_score(dy,dl.argmax(1),average='macro'); dc=softmax(dl,T_dis).max(1); dcorr=(dl.argmax(1)==dy).astype(float)
vc=softmax(vl,T_dis).max(1); vcorr=(vl.argmax(1)==vy).astype(float)
tau_dis=next(float(t) for t in np.linspace(0.5,0.999,1000) if (vc>=t).sum() and (1-vcorr[vc>=t]).mean()<=0.05)
acc=dc>=tau_dis; print('DistilBERT deploy  T=%.4f  tau=%.4f'%(T_dis,tau_dis))
print('seen macro-F1',round(d_macro,4),'| seen ECE %.4f -> %.4f'%(ece(softmax(dl,1).max(1),dcorr),ece(dc,dcorr)))
print('seen cov/risk @tau','%.1f%%'%(acc.mean()*100),round((1-dcorr[acc]).mean(),4))

## Cross-event seen(seed-42) vs unseen
Unseen = RoBERTa seed-42 Regime-B on `cyclone_idai_2019` (from nb3; unseen logits not persisted). SEEN recomputed from A4 seed-42 = the fix for nb3's seed mismatch.

In [ ]:
unseen={'caution_and_advice':0.690,'displaced_people_and_evacuations':0.750,'infrastructure_and_utility_damage':0.847,
'injured_or_dead_people':0.950,'missing_or_found_people':0.667,'not_humanitarian':0.438,'other_relevant_information':0.573,
'requests_or_urgent_needs':0.509,'rescue_volunteering_or_donation_effort':0.905,'sympathy_and_support':0.854}; unseen_macro=0.7182
for i,l in enumerate(LABELS): print(f'{l:40s} {r_f1[i]:.3f}  {unseen[l]:.3f}  {r_f1[i]-unseen[l]:+.3f}')
print(f'{"MACRO":40s} {r_macro:.3f}  {unseen_macro:.3f}  {r_macro-unseen_macro:+.3f}')
print('Unseen ECE 0.041 -> 0.061 (frozen T over-corrects); unseen selective cov 36.8%/risk 0.011 (review 63%).')

## Write metadata.json (deployment=DistilBERT, analysis=RoBERTa, held_out=cyclone_idai)

In [ ]:
meta={'model_version':'1.1.0','note':'Deployment ships DistilBERT with its own T/tau. RoBERTa is the analysis model. seed-42 canonical.',
 'canonical_label_order':LABELS,
 'training_config':{'epochs':5,'batch_size':32,'learning_rate':2e-5,'warmup_ratio':0.1,'weight_decay':0.01,'max_seq_length':128},
 'dataset':'HumAID events_set1_47K (13 events, ~43k rows)','held_out_events':['cyclone_idai_2019'],
 'deployment_model':{'encoder_id':'distilbert-base-uncased','run_id':'A1','seed':42,'imbalance_method':'plain',
   'temperature_T':round(float(T_dis),4),'review_threshold_tau':round(float(tau_dis),4),'target_risk':0.05,'seen_test_macro_f1':round(float(d_macro),4)},
 'analysis_model':{'encoder_id':'cardiffnlp/twitter-roberta-base','run_id':'A4','seed':42,'imbalance_method':'plain',
   'temperature_T':1.3509,'review_threshold_tau':0.8945,'target_risk':0.05,'seen_test_macro_f1':round(float(r_macro),4),
   'seeds_available':[42,123],'two_seed_mean_test_macro_f1':0.7642}}
json.dump(meta,open('humaid-nb1-outputs/models/transformer/metadata.json','w'),indent=2); print('metadata.json written')